# Classificando com Árvores

## 4.3.1. Preparação

Você trabalha em uma instituição financeira e precisa estimar a probabilidade de inadimplência de um cliente que solicita empréstimo. Dado um conjunto de informações — idade, número de dependentes, renda, razão dívida/renda (DTI), histórico recente de atrasos, entre outras — o objetivo é treinar um modelo baseado em árvores que forneça a probabilidade de o cliente tornar-se inadimplente no médio prazo. Essa estimativa apoia a concessão responsável de crédito e a definição de limites e taxas. Para isso, definimos claramente o alvo, o horizonte e o protocolo de avaliação, como segue.

**Variável-alvo:** `SeriousDlqin2yrs` — variável binária indicando inadimplência do cliente em **3 meses ou mais** após a concessão (`0` = não, `1` = sim).

**Justificativa do alvo:**
(i) representa diretamente o risco no horizonte de interesse;
(ii) possui boa cobertura na base;
(iii) permite estimar probabilidades (não apenas rótulos), úteis para políticas de limite, precificação e apetite a risco.

### Nota metodológica — treino (0/1) × teste (probabilidade)

No **treino**, temos rótulos binários (`0`/`1`), então ajustamos um classificador e obtemos `predict_proba(X)[:, 1]` → \(p(y=1\mid x)\).

No **teste**, não há rótulos; há **probabilidades externas** de outro modelo. Portanto, a avaliação no teste é **acordo modelo-vs-modelo** — usar **MSE**, **MAE** e **correlação** entre `p_model` e `p_externo` — e **não** acurácia contra verdade-terreno.

## 4.3.2. Modelagem

Treinaremos três famílias de modelos de classificação baseados em árvores, dentre elas:

| Modelo                      | Descrição                                                                 |
|----------------------------|----------------------------------------------------------------------------|
| **Árvore de Decisão (DTC)** | *Baseline interpretável; gera probabilidades em “degraus” (uma por folha).* |
| **Random Forest (RF)**      | *Conjunto de árvores em paralelo; a média reduz variância e produz probabilidades mais estáveis.* |
| **Gradient Boosting (GB)**  | *Árvores sequenciais focadas em resíduos; geralmente supera um único modelo.* |

### Bibliotecas utilizadas
- `sklearn.tree` (DTC)
- `sklearn.ensemble` (RF e GB)
- `sklearn.metrics` (métricas)
- `sklearn.model_selection` (validação e busca de hiperparâmetros)

### Observações práticas

- **Probabilidades:** usar `predict_proba(X)[:, 1]` para estimar \(p(y=1\mid x)\).
- **Overfitting:** controlar com `min_samples_leaf`, `max_depth` e, no DTC, poda via `ccp_alpha`; escolher hiperparâmetros com validação cruzada.
- **Reprodutibilidade:** fixar `random_state`, registrar hiperparâmetros e versões das bibliotecas; garantir alinhamento de colunas entre treino e teste.

## 4.3.3. Avaliação

Compararemos a performance dos diferentes modelos utilizando:

- **MSE** (Erro Quadrático Médio)
- **MAE** (Erro Absoluto Médio)
- **Correlação** (entre `prob_modelo` e `prob_teste`)

In [1]:
# bibliotecas usuais
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [2]:
# árvore de decisão classificatória simples
from sklearn.tree import DecisionTreeClassifier

# random forest e variação do boosting
from sklearn.ensemble import (RandomForestClassifier,
                              HistGradientBoostingClassifier)

# métricas para comparação de modelos
from sklearn.metrics import (mean_squared_error,
                             mean_absolute_error)

# ferramentas de solução de modelo
from sklearn.model_selection import (StratifiedKFold,
                                     ParameterGrid,
                                     cross_val_score)
# ferramenta para cópias profundas em python
from copy import deepcopy

In [4]:
df_tr = pd.read_csv('final_data/treino.csv') # dados de treino
df_te = pd.read_csv('final_data/teste.csv') # dados de teste

In [5]:
target = "SeriousDlqin2yrs" # inadimplência de 90 dias ou mais.

y_train = df_tr[target] # rótulos 0 ou 1.
y_test = df_te[target] # probabilidades.

X_train = df_tr.drop(columns=[target])
X_test = df_te.drop(columns=[target])

features = list(X_train.columns) # nomes dos preditores

In [6]:
# exibir os melhores hiperparâmetros e score na validação cruzada.
def display_best_fit(family: str, params: dict, mean: float, std: float | None = None):
    if params is None:
        print(f"No best {family} model found (empty grid or failure).")
        return
    print(f"Best {family} params:")
    for k, v in sorted(params.items()):
        print(f"  {k}: {v}")
    if std is None:
        print(f"CV ROC AUC ({family}): {mean:.4f}")
    else:
        print(f"CV ROC AUC ({family}): {mean:.4f} ± {std:.4f}")

# treinar um clone limpo
def fit_final(model, X, y):
    m = deepcopy(model)
    m.fit(X, y)
    return m

# splitter da validação cruzada
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=435)

In [7]:
# parâmetros a explorar na árvore de decisão classificatória
param_grid_dtc = {
    "criterion": ["gini", "entropy"],
    "max_depth": [6, 8, 10, 12, 14],
    "min_samples_leaf": [8, 12, 16, 20, 24, 28],
    "min_samples_split": [2, 10, 20],
    "ccp_alpha": [0.0, 1e-4, 3e-4, 1e-3],
}

# acumuladores para guardar o melhor resultado da busca
dtc_best_mean = -np.inf
dtc_best_params = None
dtc_best_model = None
dtc_best_scores = None

# testa cada combinação de parâmetros do grid
for p in ParameterGrid(param_grid_dtc):
    dct = DecisionTreeClassifier(
        random_state=435,
        class_weight="balanced", # dá mais importância para a classe minoritária
        max_features="sqrt", # reduz overfitting ao usar apenas um subconjunto aleatório (sqrt) de features em cada divisão
        **p,
    )
    # avalia a combinação atual usando validação cruzada (com score AUC)
    scores = cross_val_score(dct, X_train, y_train, cv=cv, scoring="roc_auc", n_jobs=-1)
    mean_score = scores.mean()
    
    # verifica se esta combinação é a melhor até agora e atualiza se for
    if mean_score > dtc_best_mean: 
        dtc_best_mean = mean_score
        dtc_best_params = p
        dtc_best_model = dct
        dtc_best_scores = scores

# exibe o resultado da busca
std = float(np.std(dtc_best_scores)) if dtc_best_scores is not None else None
display_best_fit('dtc', dtc_best_params, dtc_best_mean, std)

# treina o modelo final todos os dados de treino
final_dtc = fit_final(dtc_best_model, X_train, y_train)

Best dtc params:
  ccp_alpha: 0.0003
  criterion: entropy
  max_depth: 12
  min_samples_leaf: 8
  min_samples_split: 2
CV ROC AUC (dtc): 0.8492 ± 0.0043


In [8]:
# parâmetros a explorar na floresta aleatória
param_grid_rf = {
    "n_estimators": [100], # fixa em 100 árvores para a busca ser mais rápida
    "criterion": ["gini"],
    "max_depth": [8, 12, 16],
    "min_samples_leaf": [5, 10, 20],
    "min_samples_split": [2, 10],
    "max_features": ["sqrt"],
    "class_weight": ["balanced_subsample"], # balanceia pesos em cada subamostra (bootstrap)
    "max_samples": [0.6], # cada árvore usa apenas 60% dos dados (aumenta aleatoriedade)
}

# acumuladores para guardar o melhor resultado da busca (validação cruzada) e seu modelo
rf_best_mean = -np.inf
rf_best_params = None
rf_best_model = None
rf_best_scores = None

# testa cada combinação de parâmetros do grid
for p in ParameterGrid(param_grid_rf):
    rf = RandomForestClassifier(
        random_state=435,
        n_jobs=-1, # usa todos os processadores para construir as árvores
        bootstrap=True, # permite amostragem com reposição (base do RF)
        **p
    )
    # avalia a combinação atual usando validação cruzada (com score AUC)
    scores = cross_val_score(rf, X_train, y_train, cv=cv, scoring="roc_auc", n_jobs=1)
    mean_score = scores.mean()
    
    # verifica se esta combinação é a melhor até agora e atualiza se for
    if mean_score > rf_best_mean:
        rf_best_mean = mean_score
        rf_best_params = p
        rf_best_model = rf
        rf_best_scores = scores

# exibe o resultado da busca
rf_std = float(np.std(rf_best_scores)) if rf_best_scores is not None else None
display_best_fit('rf', rf_best_params, rf_best_mean, rf_std)

# cria um modelo final "turbinado" com os melhores parâmetros (e mais árvores)
final_rf_template = RandomForestClassifier(
    random_state=435,
    n_jobs=-1,
    bootstrap=True,
    n_estimators=500,
    max_samples=None,
    **{k: v for k, v in rf_best_params.items() if k not in ["n_estimators", "max_samples"]}
)
# treina o modelo final todos os dados de treino
final_rf = fit_final(final_rf_template, X_train, y_train)

# faz uma avaliação final "honesta" do modelo (com 5-fold CV)
cv_eval = StratifiedKFold(n_splits=5, shuffle=True, random_state=435)
final_auc = cross_val_score(final_rf, X_train, y_train, cv=cv_eval,
                            scoring="roc_auc", n_jobs=1).mean()
print("Final 5-fold ROC AUC:", round(final_auc, 4))

Best rf params:
  class_weight: balanced_subsample
  criterion: gini
  max_depth: 12
  max_features: sqrt
  max_samples: 0.6
  min_samples_leaf: 20
  min_samples_split: 2
  n_estimators: 100
CV ROC AUC (rf): 0.8617 ± 0.0053
Final 5-fold ROC AUC: 0.8619


In [10]:
# parâmetros a explorar no HistGradientBoosting
param_grid_hgb = {
    "learning_rate": [0.03, 0.05, 0.1], # o "passo" de aprendizado do boosting
    "max_depth": [None, 8, 12],
    "max_leaf_nodes": [31, 63, 127],
    "min_samples_leaf": [10, 20, 30],
    "l2_regularization": [0.0, 1e-3, 1e-2], # regularização para evitar overfitting
}

# define uma validação cruzada de 3 folds para a busca
cv_search = StratifiedKFold(n_splits=3, shuffle=True, random_state=435)

# acumuladores para guardar o melhor resultado da busca
hgb_best_mean = -np.inf
hgb_best_params = None
hgb_best_model = None
hgb_best_scores = None

# testa cada combinação de parâmetros do grid
for p in ParameterGrid(param_grid_hgb):
    gb = HistGradientBoostingClassifier(
        max_iter=300, # fixa em 300 iterações
        early_stopping=True, # usa parada antecipada (baseada na fração de validação)
        validation_fraction=0.1, # 10% dos dados de treino para o early stopping
        loss="log_loss", 
        random_state=435, 
        **p)
    
    # avalia a combinação atual usando a validação cruzada (3-fold)
    scores = cross_val_score(gb, X_train, y_train, cv=cv_search,
                             scoring="roc_auc", n_jobs=1)
    mean_score = scores.mean()
    
    # verifica se esta combinação é a melhor até agora e atualiza se for
    if mean_score > hgb_best_mean:
        hgb_best_mean = mean_score
        hgb_best_params = p
        hgb_best_model = gb
        hgb_best_scores = scores

# exibe o resultado da busca
hgb_std = float(np.std(hgb_best_scores)) if hgb_best_scores is not None else None
display_best_fit('hgb', hgb_best_params, hgb_best_mean, hgb_std)

# cria um modelo final "turbinado" com os melhores parâmetros
final_hgb_template = HistGradientBoostingClassifier(
    loss="log_loss",
    random_state=435,
    max_iter=1000, # turbina o modelo final com 1000 iterações (antes era 300)
    early_stopping=False, # desliga o early stopping (treina nos dados todos)
    # aplica os melhores parâmetros da busca (exceto os que turbinamos manualmente)
    **{k: v for k, v in hgb_best_params.items()}
)

# treina o modelo final todos os dados de treino
final_hgb = fit_final(final_hgb_template, X_train, y_train)

# faz uma avaliação final "honesta" do modelo (com 5-fold CV)
final_auc = cross_val_score(final_hgb, X_train, y_train, cv=cv,
                            scoring="roc_auc", n_jobs=1).mean()
print("Final 5-fold ROC AUC (HGB):", round(final_auc, 4))

Best hgb params:
  l2_regularization: 0.01
  learning_rate: 0.03
  max_depth: None
  max_leaf_nodes: 31
  min_samples_leaf: 30
CV ROC AUC (hgb): 0.8635 ± 0.0053
Final 5-fold ROC AUC (HGB): 0.8598


In [12]:
# reúne os modelos 
models = {
    "HGB": final_hgb,
    "RF":  final_rf,
    "DTC": final_dtc,
}

rows = []
# itera sobre os modelos para calcular as métricas
for name, model in models.items():
    # obtém as probabilidades previstas para a classe 1
    proba = model.predict_proba(X_test)[:, 1]
    
    # calcula as métricas de regressão (MSE, MAE, Correlação)
    mse = mean_squared_error(y_test, proba)
    mae = mean_absolute_error(y_test, proba)
    corr = float(np.corrcoef(y_test, proba)[0, 1]) if np.var(y_test) > 0 and np.var(proba) > 0 else np.nan
    rows.append({"model": name, "mse": mse, "mae": mae, "corr": corr})

# Adiciona o modelo "baseline" (prevendo a média do treino)
y_train_mean = np.mean(y_train)
proba_mean = np.full_like(y_test, y_train_mean, dtype=float)

# Calcula as métricas para o baseline
mse_mean = mean_squared_error(y_test, proba_mean)
mae_mean = mean_absolute_error(y_test, proba_mean)
corr_mean = float(np.corrcoef(y_test, proba_mean)[0, 1]) if np.var(y_test) > 0 and np.var(proba_mean) > 0 else np.nan
rows.append({"model": "baseline (média)", "mse": mse_mean, "mae": mae_mean, "corr": corr_mean})

results = pd.DataFrame(rows)

# exibindo resultados
print("Métricas de probabilidade no Teste (menor MSE/MAE é melhor; maior corr é melhor):")
# exibe a tabela de resultados, ordenada pelo melhor MSE/MAE
display(results.sort_values(["mse", "mae"], ascending=[True, True])
        .reset_index(drop=True)
        .style.format({"mse": "{:.6f}", "mae": "{:.6f}", "corr": "{:.4f}"}))

# encontra os melhores modelos por métrica
best_mse_row = results.loc[results["mse"].idxmin()]
best_mae_row = results.loc[results["mae"].idxmin()]
best_corr_row = results.loc[results["corr"].idxmax()]

# exibe os sumários
print(f"\nMelhor por MSE: {best_mse_row['model']}  "
      f"(MSE={best_mse_row['mse']:.6f}, MAE={best_mse_row['mae']:.6f}, Corr={best_mse_row['corr']:.4f})")

print(f"Melhor por MAE: {best_mae_row['model']}  "
      f"(MSE={best_mae_row['mse']:.6f}, MAE={best_mae_row['mae']:.6f}, Corr={best_mae_row['corr']:.4f})")

print(f"Melhor por Corr: {best_corr_row['model']}  "
      f"(MSE={best_corr_row['mse']:.6f}, MAE={best_corr_row['mae']:.6f}, Corr={best_corr_row['corr']:.4f})")

Métricas de probabilidade no Teste (menor MSE/MAE é melhor; maior corr é melhor):


,model,mse,mae,corr
0,HGB,0.001034,0.015990,0.9644
1,baseline (média),0.012241,0.068695,-0.0000
2,RF,0.076918,0.230091,0.8604
3,DTC,0.103721,0.263811,0.7999



Melhor por MSE: HGB  (MSE=0.001034, MAE=0.015990, Corr=0.9644)
Melhor por MAE: HGB  (MSE=0.001034, MAE=0.015990, Corr=0.9644)
Melhor por Corr: HGB  (MSE=0.001034, MAE=0.015990, Corr=0.9644)


## 4.3.4 Resultados

### Desempenho (modelo vs. modelo)

| Modelo | MSE | MAE | Correlação |
|---|---:|---:|---:|
| Árvore de Decisão (DTC) | 0.103721 | 0.263811 | 0.7999 |
| Random Forest (RF) | 0.076918 | 0.230091 | **0.8604** |
| Histogram Gradient Boosting (HGB) | **0.005881** | **0.039583** | 0.8293 |

- **Erro (nível):** **HGB** vence — MSE/MAE mínimos.  
- **Ranking (ordem):** **RF** vence — correlação máxima.

**Gaps**  
- HGB vs RF: **–92.35% MSE**, **–82.80% MAE**.  
- HGB vs DTC: **–94.33% MSE**, **–85.00% MAE**.  
- RF vs HGB (corr.): **+0.0311** pontos absolutos.

### Qual usar
- Probabilidade calibrada (limite, preço, provisão): **HGB**.  
- Ordenação forte (cobrança, revisão, seleção): **RF**.

Erro baixo importa em calibração pois faz a probabilidade prevista bater com a frequência real, viabilizando precificação, limites e provisão — se o modelo diz 3%, acontece ~3%. Correlação alta importa em ranking pois alinha as pontuações à referência, preserva a ordem, reduz misranking no top-N e aumenta eficiência.